# 01. FBref Data Ingestion

**Stage:** Ingestion  
**Inputs:** `soccerdata` FBref and ClubElo readers, EPL seasons from config  
**Outputs:** `data/raw/{season}/fixtures.parquet`, `data/raw/{season}/match_stats.parquet`

This notebook loads fixtures and team match statistics. Player availability is deferred to Task 2.1 because retrieving player match pages one fixture at a time is expensive and minutes alone cannot prove an absence reason.

In [ ]:
# Load configuration and imports
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.ingestion.soccerdata_client import SoccerDataClient, SoccerDataConfig

config = load_config()
season = config["data"]["seasons"][0]
league_id = config["data"]["default_league_id"]
ingestion_config = config["ingestion"]["soccerdata"]
print(f"Season: {season}")
print(f"League ID: {league_id}")
fixtures = None
match_stats = None


In [ ]:
# NBVAL_SKIP
client = SoccerDataClient(config=SoccerDataConfig(league=ingestion_config["league"]))
try:
    fixtures = client.fetch_fixtures(season=season, league_id=league_id)
    print(f"Fetched {len(fixtures)} fixtures from FBref")
except Exception as exc:
    print(f"Error fetching fixtures: {exc}")
    fixtures = None

In [ ]:
# NBVAL_SKIP
# Fetch team match statistics once; soccerdata returns the full season table.
try:
    if fixtures is not None and len(fixtures) > 0:
        match_stats = client.fetch_all_match_stats(season=season)
        print(f"Fetched {len(match_stats)} team match-stat rows")
    else:
        match_stats = None
except Exception as exc:
    print(f"Error fetching match stats: {exc}")
    match_stats = None

In [ ]:
# Player availability is deferred to Task 2.1.
# Do not retrieve player match pages for every fixture during base ingestion.
player_minutes = None
print("Player availability retrieval deferred to the absence-classification stage.")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Clean season string for safe directory naming
season_clean = season.replace("/", "-")

# Construct the absolute path directly from the root
output_dir = PROJECT_ROOT / "data" / "raw" / season_clean
output_dir.mkdir(parents=True, exist_ok=True)

if fixtures is not None and len(fixtures) > 0:
    fixtures.to_parquet(output_dir / "fixtures.parquet", index=False)
    print(f"Saved {len(fixtures)} fixtures")
if match_stats is not None and len(match_stats) > 0:
    match_stats.to_parquet(output_dir / "match_stats.parquet", index=False)
    print(f"Saved {len(match_stats)} team match-stat rows")

In [ ]:
if fixtures is not None and len(fixtures) > 0:
    print("Sample fixtures:")
    display(fixtures.head())
if match_stats is not None and len(match_stats) > 0:
    print("Sample team match statistics:")
    display(match_stats.head())

In [ ]:
print("=== Data Validation ===")
if fixtures is not None and len(fixtures) > 0:
    required_columns = ["fixture_id", "home_team", "away_team", "date"]
    missing_columns = [column for column in required_columns if column not in fixtures.columns]
    print(f"Fixtures: {len(fixtures)} rows")
    print(f"Missing required columns: {missing_columns}")
if match_stats is not None and len(match_stats) > 0:
    print(f"Team match stats: {len(match_stats)} rows")
print("=== Ingestion Complete ===")